# QSAR: machine-learning prediction of compound activity

End-to-end pipeline: **ChEMBL data -> cleaning -> Morgan fingerprints ->
RandomForest QSAR model -> evaluation -> substructure interpretation**.

The code cells are auto-generated from `scripts/` (see
`scripts/06_build_notebook.py`), so the notebook never drifts from the
runnable pipeline. Set the environment variable `QSAR_TAG` to `egfr`
(default) or `bace` to switch datasets.


## Step 1 - Download bioactivity data from ChEMBL

Query the ChEMBL REST API for **EGFR (CHEMBL203)** activities: `standard_type=IC50`, `standard_units=nM`, binding/functional assays.

The API was intermittently returning HTTP 500 during this project, so the downloader retries every page, saves progress after each page, and skips poisoned windows (see `scripts/01d_download_incremental.py`). A ready-made fallback source (MoleculeNet BACE-1) is provided in `scripts/01alt_download_bace.py` - set `QSAR_TAG=bace` to use it.

In [ ]:
"""Step 1 (incremental): resume-friendly EGFR download with skip-ahead.

Learnings from earlier runs:
  * API is intermittently 500 (bad backend nodes) -> retry each page.
  * Offset ~2500 is consistently poisoned (a specific record seems to crash
    the serializer) -> after 8 failed tries, SKIP that 500-row window and
    keep going; partial data is fine (we need 1000+ compounds, not all).
  * Rows were lost when the script died mid-run -> append+flush every page.

Fetches assay_type=B first, then F (separate result sets also dodge the
poisoned record), stops early once TARGET_ROWS raw rows are collected.
"""
import os, time
import pandas as pd
import requests

TARGET_ID = "CHEMBL203"
API = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
OUT_DIR = os.path.join(os.path.dirname(__file__), "..", "data", "raw")
OUT_CSV = os.path.join(OUT_DIR, "egfr_chembl_activities.csv")
TARGET_ROWS = 8000          # early stop: plenty for 1000+ unique compounds
PAGE = 500
MAX_TRIES_PER_PAGE = 8

COLUMNS = [
    "molecule_chembl_id", "canonical_smiles", "standard_type",
    "standard_value", "standard_units", "pchembl_value", "assay_type",
    "assay_description", "target_pref_name", "bao_label", "document_chembl_id",
]


def get_page(params, max_tries=MAX_TRIES_PER_PAGE):
    for t in range(1, max_tries + 1):
        try:
            r = requests.get(API, params=params, timeout=90)
            if r.status_code == 200:
                return r.json()
            print(f"    HTTP {r.status_code} (try {t})", flush=True)
        except Exception as e:
            print(f"    {e} (try {t})", flush=True)
        time.sleep(min(8 * t, 45))
    return None


def fetch_assay_type(assay_type, total_rows):
    offset, skipped = 0, []
    while total_rows < TARGET_ROWS:
        params = {"target_chembl_id": TARGET_ID, "standard_type": "IC50",
                  "standard_units": "nM", "assay_type": assay_type,
                  "limit": PAGE, "offset": offset}
        js = get_page(params)
        if js is None:
            print(f"  SKIP window [{offset}, {offset + PAGE})", flush=True)
            skipped.append((offset, offset + PAGE))
            offset += PAGE
            if len(skipped) > 6:
                print("  too many skips, moving on.", flush=True)
                break
            continue
        batch = js.get("activities", [])
        if not batch:
            break
        df = pd.DataFrame([{c: a.get(c) for c in COLUMNS} for a in batch])
        df.to_csv(OUT_CSV, mode="a", header=not os.path.exists(OUT_CSV), index=False)
        total_rows += len(batch)
        print(f"  {assay_type}: +{len(batch)} (total {total_rows}, offset {offset})", flush=True)
        offset += len(batch)
        if not js.get("page_meta", {}).get("next"):
            break
        time.sleep(0.3)
    return total_rows, skipped


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)
    total, all_skipped = 0, []
    for at in ["B", "F"]:
        print(f"Fetching assay_type={at} ...", flush=True)
        total, sk = fetch_assay_type(at, total)
        all_skipped += sk
        if total >= TARGET_ROWS:
            break
    print(f"DONE: {total} raw rows -> {OUT_CSV}; skipped windows: {all_skipped}", flush=True)


if __name__ == "__main__":
    main()


## Step 2 - Data cleaning & pIC50 transform

Keep nM rows, drop missing SMILES/IC50, convert to pIC50 (`-log10(IC50 * 1e-9)`) and collapse duplicate compounds to the **median** pIC50.

In [ ]:
"""Step 2: Clean the raw ChEMBL activity data.

What / why:
  1. Keep only rows whose IC50 unit is exactly 'nM'  -> comparable numbers.
  2. Drop rows with missing standard_value (IC50) or empty canonical_smiles.
  3. Drop non-positive IC50 values (a 0 breaks the log transform).
  4. Convert IC50 (nM) -> pIC50 = -log10(IC50 * 1e-9).
     Why: IC50 spans many orders of magnitude (1 nM ... 100 uM). Raw values
     are heavy-tailed and would dominate model training; pIC50 is roughly
     normal and is THE standard target transform in QSAR.
  5. One compound may be measured in many papers/assays -> duplicates.
     Group by canonical_smiles and take the MEDIAN pIC50 (robust to outliers).
"""
import os
import numpy as np
import pandas as pd

BASE = os.path.join(os.path.dirname(__file__), "..")
TAG = os.environ.get("QSAR_TAG", "egfr")  # which dataset: egfr | bace
IN_CSV = os.path.join(BASE, "data", "raw", f"{TAG}_activities.csv")
OUT_CSV = os.path.join(BASE, "data", "processed", f"{TAG}_pic50_clean.csv")


def main():
    df = pd.read_csv(IN_CSV)
    print(f"Loaded {len(df)} raw rows.")

    # 1. unit must be nM
    df = df[df["standard_units"] == "nM"].copy()
    print(f"After unit == nM: {len(df)}")

    # 2. drop missing IC50 / SMILES
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
    df = df.dropna(subset=["standard_value", "canonical_smiles"])
    df = df[df["canonical_smiles"].str.strip() != ""]
    print(f"After dropping missing IC50/SMILES: {len(df)}")

    # 3. non-positive IC50 is physically meaningless for the log transform
    df = df[df["standard_value"] > 0].copy()
    print(f"After dropping IC50 <= 0: {len(df)}")

    # 4. IC50 (nM) -> pIC50
    df["pic50"] = -np.log10(df["standard_value"] * 1e-9)

    # sanity cross-check against ChEMBL's own pchembl_value (should match)
    chk = df.dropna(subset=["pchembl_value"])
    agree = np.isclose(chk["pic50"], chk["pchembl_value"].astype(float), atol=0.02).mean()
    print(f"Sanity check: pIC50 matches ChEMBL pchembl_value for {agree:.1%} of rows.")

    # 5. deduplicate by canonical SMILES -> median pIC50
    n_dup = df.duplicated(subset=["canonical_smiles"]).sum()
    target_name = df["target_pref_name"].dropna().iloc[0] if df["target_pref_name"].notna().any() else TAG
    clean = (
        df.groupby("canonical_smiles", as_index=False)
        .agg(
            pic50=("pic50", "median"),
            ic50_nm_median=("standard_value", "median"),
            n_measurements=("pic50", "size"),
            molecule_chembl_id=("molecule_chembl_id", "first"),
        )
        .reset_index(drop=True)
    )
    clean["target"] = target_name
    print(f"Target: {target_name}")
    print(f"Removed {n_dup} duplicate rows -> {len(clean)} unique compounds.")

    print("\npIC50 distribution:")
    print(clean["pic50"].describe())

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    clean.to_csv(OUT_CSV, index=False)
    print(f"\nSaved cleaned data -> {os.path.abspath(OUT_CSV)}")


if __name__ == "__main__":
    main()


## Step 3 - Morgan fingerprints (RDKit)

Each SMILES becomes a 2048-bit Morgan fingerprint (radius 2). Invalid SMILES that RDKit cannot parse are dropped.

In [ ]:
"""Step 3: Featurize SMILES -> Morgan fingerprints (RDKit).

Why Morgan fingerprints:
  A molecule is a graph; ML needs numbers. Morgan fingerprint (radius=2,
  ECFP4-like) encodes every atom's circular neighbourhood up to 2 bonds as
  a hashed 2048-bit vector. Structurally similar molecules -> similar
  fingerprints, which is exactly the signal a similarity-based ML model
  (random forest / XGBoost) can learn from.
  Invalid SMILES (rare in ChEMBL, but they exist) make RDKit return None;
  those compounds are dropped.
Output: data/processed/egfr_fingerprints.npz  (X, y, smiles)
"""
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")  # silence per-molecule parse warnings

BASE = os.path.join(os.path.dirname(__file__), "..")
TAG = os.environ.get("QSAR_TAG", "egfr")  # which dataset: egfr | bace
IN_CSV = os.path.join(BASE, "data", "processed", f"{TAG}_pic50_clean.csv")
OUT_NPZ = os.path.join(BASE, "data", "processed", f"{TAG}_fingerprints.npz")

RADIUS = 2
NBITS = 2048


def main():
    df = pd.read_csv(IN_CSV)
    print(f"Loaded {len(df)} cleaned compounds.")

    X = np.zeros((len(df), NBITS), dtype=np.uint8)
    y = df["pic50"].to_numpy(dtype=np.float32)
    valid_idx = []

    for i, smi in enumerate(df["canonical_smiles"]):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=NBITS)
        X[i] = np.frombuffer(fp.ToBitString().encode(), dtype=np.uint8) - ord("0")
        valid_idx.append(i)

    n_invalid = len(df) - len(valid_idx)
    X = X[valid_idx]
    y = y[valid_idx]
    smiles = np.asarray(df["canonical_smiles"].to_numpy()[valid_idx], dtype=str)
    target = str(df["target"].iloc[0]) if "target" in df.columns else TAG
    print(f"Dropped {n_invalid} invalid SMILES -> X shape {X.shape}, y shape {y.shape}")

    np.savez_compressed(OUT_NPZ, X=X, y=y, smiles=smiles, target=np.asarray(target, dtype=str))
    print(f"Saved feature matrix -> {os.path.abspath(OUT_NPZ)}")


if __name__ == "__main__":
    main()


## Steps 4-6 - Split, train, evaluate

Random 80/20 split vs **scaffold split** (Bemis-Murcko scaffolds kept whole - simulates predicting new chemotypes). RandomForest with GridSearchCV; metrics: R2 / RMSE / MAE + scatter plots.

In [ ]:
"""Steps 4-6: Split data, train RandomForest (with GridSearchCV), evaluate.

Step 4 - two split strategies:
  * random split: train_test_split(80/20, random_state=42).
    EASY, but near-identical analogues can land on both sides -> score
    is optimistic.
  * scaffold split: group molecules by Bemis-Murcko scaffold and put
    whole scaffolds into train or test. Simulates predicting truly NEW
    chemotypes -> the honest number. Implemented with RDKit only (no
    DeepChem dependency).

Step 5 - model:
  RandomForestRegressor with GridSearchCV over
  n_estimators / max_depth / min_samples_split (5-fold CV on training set).

Step 6 - evaluation:
  R2, RMSE, MAE on the test set + predicted-vs-actual scatter plot.
"""
import json
import os
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

BASE = os.path.join(os.path.dirname(__file__), "..")
TAG = os.environ.get("QSAR_TAG", "egfr")  # which dataset: egfr | bace
IN_NPZ = os.path.join(BASE, "data", "processed", f"{TAG}_fingerprints.npz")
FIG_DIR = os.path.join(BASE, "figures")
MODEL_DIR = os.path.join(BASE, "models")
METRICS_JSON = os.path.join(BASE, "results", f"metrics_{TAG}.json")
RANDOM_STATE = 42

PARAM_GRID = {
    "n_estimators": [300, 500],
    "max_depth": [None, 20, 40],
    "min_samples_split": [2, 5],
}


def scaffold_split(smiles, test_size=0.2):
    """Assign whole Bemis-Murcko scaffolds to train/test (test ~= test_size)."""
    scaffolds = {}
    for i, smi in enumerate(smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            scaf = smi  # fallback: treat molecule as its own scaffold
        else:
            scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        scaffolds.setdefault(scaf, []).append(i)

    # big scaffolds first -> test set fills up to roughly test_size
    groups = sorted(scaffolds.values(), key=len, reverse=True)
    n_test = int(len(smiles) * test_size)
    test_idx, train_idx, n = [], [], 0
    for g in groups:
        if n < n_test:
            test_idx.extend(g)
            n += len(g)
        else:
            train_idx.extend(g)
    return np.array(train_idx), np.array(test_idx)


def evaluate(model, X_test, y_test, tag):
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = mean_absolute_error(y_test, pred)
    print(f"[{tag}]  R2 = {r2:.3f}   RMSE = {rmse:.3f}   MAE = {mae:.3f}")
    return {"split": tag, "r2": float(r2), "rmse": rmse, "mae": float(mae)}, pred


def scatter(y_test, pred, tag, path, target):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, pred, s=10, alpha=0.4, edgecolors="none")
    lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="ideal")
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"{target} QSAR - RandomForest ({tag} split)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def run_split(X, y, smiles, train_idx, test_idx, tag, fig_path, target):
    print(f"\n=== {tag} split: train={len(train_idx)}, test={len(test_idx)} ===")
    gs = GridSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        PARAM_GRID,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        verbose=1,
    )
    gs.fit(X[train_idx], y[train_idx])
    print(f"Best params: {gs.best_params_}  (CV R2 = {gs.best_score_:.3f})")
    metrics, pred = evaluate(gs.best_estimator_, X[test_idx], y[test_idx], tag)
    scatter(y[test_idx], pred, tag, fig_path, target)
    return metrics, gs.best_estimator_, pred


def main():
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(METRICS_JSON), exist_ok=True)

    d = np.load(IN_NPZ)
    X, y, smiles = d["X"], d["y"], d["smiles"]
    target = str(d["target"]) if "target" in d.files else TAG
    print(f"Target: {target} | Feature matrix: X={X.shape}, y range [{y.min():.2f}, {y.max():.2f}]")

    # ---- random split ----
    tr, te = train_test_split(
        np.arange(len(X)), test_size=0.2, random_state=RANDOM_STATE
    )
    m_rand, best_rand, _ = run_split(
        X, y, smiles, tr, te, "random",
        os.path.join(FIG_DIR, f"pred_vs_actual_random_{TAG}.png"), target
    )
    joblib.dump(best_rand, os.path.join(MODEL_DIR, f"rf_random_split_{TAG}.joblib"))

    # ---- scaffold split ----
    tr_s, te_s = scaffold_split(smiles, test_size=0.2)
    m_scaf, best_scaf, _ = run_split(
        X, y, smiles, tr_s, te_s, "scaffold",
        os.path.join(FIG_DIR, f"pred_vs_actual_scaffold_{TAG}.png"), target
    )
    joblib.dump(best_scaf, os.path.join(MODEL_DIR, f"rf_scaffold_split_{TAG}.joblib"))

    with open(METRICS_JSON, "w") as f:
        json.dump({"target": target, "tag": TAG, "random": m_rand, "scaffold": m_scaf,
                   "best_params_random": best_rand.get_params()}, f, indent=2)
    print(f"\nMetrics saved -> {os.path.abspath(METRICS_JSON)}")


if __name__ == "__main__":
    main()


## Step 6 (advanced) - What do the top bits mean?

Map the most important fingerprint bits back to the molecular substructures that set them (RDKit bitInfo + PathToSubmol).

In [ ]:
"""Step 6 (advanced): map the most important fingerprint bits to substructures.

Random forests expose feature_importances_ per bit. A fingerprint bit alone is
opaque; here we find, for each top bit, an example molecule in the dataset
where the bit is set and reconstruct the exact substructure (atom environment)
that hashed into that bit, using RDKit's bitInfo machinery.
Output:
  results/top_bits.csv          bit, substructure SMILES, #molecules carrying it
  figures/top_bits.png          grid drawing of those substructures
"""
import os
import numpy as np
import pandas as pd
import joblib
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

BASE = os.path.join(os.path.dirname(__file__), "..")
TAG = os.environ.get("QSAR_TAG", "egfr")  # which dataset: egfr | bace
IN_NPZ = os.path.join(BASE, "data", "processed", f"{TAG}_fingerprints.npz")
MODEL = os.path.join(BASE, "models", f"rf_random_split_{TAG}.joblib")
OUT_CSV = os.path.join(BASE, "results", f"top_bits_{TAG}.csv")
OUT_PNG = os.path.join(BASE, "figures", f"top_bits_{TAG}.png")
TOP_N = 9
RADIUS = 2
NBITS = 2048


def substructure_for_bit(mol, atom_idx, radius):
    """Extract the atom environment that set a fingerprint bit.

    Returns (smiles, drawable_submol). Uses PathToSubmol (a real sub-molecule)
    instead of MolFragmentToSmiles, whose output often fails to re-parse
    (fragment valences don't round-trip).
    """
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    submol = Chem.PathToSubmol(mol, env)
    return Chem.MolToSmiles(submol), submol


def main():
    model = joblib.load(MODEL)
    d = np.load(IN_NPZ)
    smiles = d["smiles"]

    importances = model.feature_importances_
    top_bits = np.argsort(importances)[::-1][:TOP_N]
    print("Top bits:", top_bits.tolist())

    # single pass: count carriers per bit AND keep the largest example
    # environment found (small fragments like a bare "O" are valid bits but
    # meaningless to show)
    counts = {int(b): 0 for b in top_bits}
    best = {int(b): (0, None, None) for b in top_bits}  # bit -> (n_heavy, smi, mol)
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        info = {}
        AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=NBITS, bitInfo=info)
        for bit in top_bits:
            bit = int(bit)
            if bit not in info:
                continue
            counts[bit] += 1
            na_cur = best[bit][0]
            if na_cur >= 6:
                continue
            for atom_idx, r in info[bit]:
                sub_smi, submol = substructure_for_bit(mol, atom_idx, r)
                na = submol.GetNumHeavyAtoms()
                if na > best[bit][0]:
                    best[bit] = (na, sub_smi, submol)
                if best[bit][0] >= 6:
                    break

    rows, mols, legends = [], [], []
    for bit in top_bits:
        bit = int(bit)
        na, sub_smi, submol = best[bit]
        rows.append({"bit": bit,
                     "importance": float(importances[bit]),
                     "substructure_smiles": sub_smi,
                     "n_heavy_atoms_in_example": na,
                     "n_molecules_with_bit": counts[bit]})
        if submol is not None and na >= 2:
            mols.append(submol)
            legends.append(f"bit {bit} (imp={importances[bit]:.3f})")
        print(f"bit {bit:4d}  importance={importances[bit]:.4f}  "
              f"in {counts[bit]:5d} mols  e.g. {sub_smi}")

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

    if mols:
        img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(320, 320),
                                   legends=legends)
        img.save(OUT_PNG)
        print(f"Saved -> {os.path.abspath(OUT_CSV)} and {os.path.abspath(OUT_PNG)}")


if __name__ == "__main__":
    main()


## QC - distributions, outliers, honest importance

Three sanity checks: (1) test-set pIC50 distributions of both splits (RMSE is scale-dependent), (2) audit of ultra-potent records (pIC50 > 9) by assay/source, (3) permutation importance as an unbiased cross-check of MDI, which is inflated for correlated fingerprint bits.

In [ ]:
"""QC checks beyond the headline metrics (run after 04_train_and_evaluate.py).

Three checks:
  1. Test-set pIC50 distributions of the random vs scaffold split. RMSE is
     scale-dependent: if the random test set spans a wider potency range, its
     RMSE can be larger even when R2 is identical - worth showing explicitly.
  2. Ultra-potent outliers (pIC50 > 9, i.e. IC50 < 1 nM). Values like this are
     often assay detection limits, unit mislabels or cross-assay artefacts
     rather than true affinity. Group them by assay / document to see whether
     a few suspicious sources dominate.
  3. Permutation importance vs MDI. RandomForest's built-in (MDI) importance
     is inflated for correlated features - and 2048 fingerprint bits contain
     many chemically equivalent ones. Permutation importance on the held-out
     test set is the honest cross-check before telling the substructure story.

Splits are reconstructed exactly (same random_state / deterministic scaffold
function as scripts/04_train_and_evaluate.py).
"""
import importlib.util
import os
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

BASE = os.path.join(os.path.dirname(__file__), "..")
TAG = os.environ.get("QSAR_TAG", "egfr")
NPZ = os.path.join(BASE, "data", "processed", f"{TAG}_fingerprints.npz")
RAW = os.path.join(BASE, "data", "raw", f"{TAG}_activities.csv")
MODEL = os.path.join(BASE, "models", f"rf_random_split_{TAG}.joblib")
FIG_HIST = os.path.join(BASE, "figures", f"testset_hist_{TAG}.png")
FIG_IMP = os.path.join(BASE, "figures", f"importance_mdi_vs_perm_{TAG}.png")
OUT_PERM = os.path.join(BASE, "results", f"permutation_importance_{TAG}.csv")
RANDOM_STATE = 42


def load_scaffold_split_func():
    spec = importlib.util.spec_from_file_location(
        "mod04", os.path.join(BASE, "scripts", "04_train_and_evaluate.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.scaffold_split


def check_testset_distributions(X, y, smiles):
    tr_r, te_r = train_test_split(np.arange(len(X)), test_size=0.2,
                                  random_state=RANDOM_STATE)
    tr_s, te_s = load_scaffold_split_func()(smiles, test_size=0.2)
    yr, ys = y[te_r], y[te_s]
    print("== 1. test-set pIC50 distributions ==")
    for name, v in [("random", yr), ("scaffold", ys)]:
        print(f"  {name:9s} n={len(v):4d}  mean={v.mean():.2f}  std={v.std():.2f}  "
              f"min={v.min():.2f}  max={v.max():.2f}")
    plt.figure(figsize=(7, 4.5))
    bins = np.linspace(min(yr.min(), ys.min()), max(yr.max(), ys.max()), 31)
    plt.hist(yr, bins=bins, alpha=0.55, label=f"random (std={yr.std():.2f})", density=True)
    plt.hist(ys, bins=bins, alpha=0.55, label=f"scaffold (std={ys.std():.2f})", density=True)
    plt.xlabel("pIC50"); plt.ylabel("density"); plt.legend()
    plt.title(f"Test-set pIC50 distribution - {TAG}")
    plt.tight_layout(); plt.savefig(FIG_HIST, dpi=150); plt.close()
    print(f"  saved -> {FIG_HIST}")


def check_ultra_potent():
    print("\n== 2. ultra-potent records (pIC50 > 9, IC50 < 1 nM) ==")
    df = pd.read_csv(RAW)
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
    df = df.dropna(subset=["standard_value", "canonical_smiles"])
    df = df[df["standard_value"] > 0]
    df["pic50"] = -np.log10(df["standard_value"] * 1e-9)
    hot = df[df["pic50"] > 9]
    print(f"  {len(hot)} / {len(df)} raw rows ({len(hot)/len(df):.1%}) have pIC50 > 9")
    if len(hot) == 0:
        return
    for col in ["assay_description", "document_chembl_id"]:
        if col in hot.columns and hot[col].notna().any():
            print(f"  top sources by {col}:")
            print(hot.groupby(col).size().sort_values(ascending=False).head(5).to_string())
    print(f"  IC50 nM range of these rows: {hot['standard_value'].min():.4f} .. {hot['standard_value'].max():.4f}")


def check_importance(X, y, smiles):
    print("\n== 3. MDI vs permutation importance (random split) ==", flush=True)
    model = joblib.load(MODEL)
    tr, te = train_test_split(np.arange(len(X)), test_size=0.2,
                              random_state=RANDOM_STATE)
    X_te, y_te = X[te], y[te]
    mdi = model.feature_importances_

    # Manual permutation importance over the top-K MDI bits only (sklearn's
    # permutation_importance would permute all 2048 columns; we only care
    # about the ranking near the top, and the model still sees full-width X).
    from sklearn.metrics import r2_score
    K, N_REP = 100, 3
    baseline = r2_score(y_te, model.predict(X_te))
    rng = np.random.RandomState(RANDOM_STATE)
    top_bits = np.argsort(mdi)[::-1][:K]
    rows = []
    for b in top_bits:
        drops = []
        for _ in range(N_REP):
            Xp = X_te.copy()
            Xp[:, b] = rng.permutation(Xp[:, b])
            drops.append(baseline - r2_score(y_te, model.predict(Xp)))
        rows.append({"bit": int(b), "mdi": float(mdi[b]),
                     "perm_mean": float(np.mean(drops)),
                     "perm_std": float(np.std(drops))})
    df = pd.DataFrame(rows).sort_values("mdi", ascending=False).reset_index(drop=True)
    df.to_csv(OUT_PERM, index=False)
    print(df.head(12).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    from scipy.stats import spearmanr
    rho = spearmanr(df["mdi"].rank(), df["perm_mean"].rank()).statistic
    print(f"  Spearman rank corr (MDI vs perm, top-{K} bits): {rho:.3f}", flush=True)

    plot = df.head(15).iloc[::-1]
    ypos = np.arange(len(plot))
    plt.figure(figsize=(8, 5))
    plt.barh(ypos - 0.2, plot["mdi"], height=0.38, label="MDI (built-in)")
    plt.barh(ypos + 0.2, plot["perm_mean"], height=0.38,
             xerr=plot["perm_std"], label="permutation (test set)")
    plt.yticks(ypos, [f"bit {b}" for b in plot["bit"]])
    plt.xlabel("importance"); plt.legend()
    plt.title(f"MDI vs permutation importance - {TAG} (top-{K} MDI bits)")
    plt.tight_layout(); plt.savefig(FIG_IMP, dpi=150); plt.close()
    print(f"  saved -> {OUT_PERM} and {FIG_IMP}", flush=True)


def main():
    d = np.load(NPZ)
    X, y, smiles = d["X"], d["y"], d["smiles"]
    check_testset_distributions(X, y, smiles)
    check_ultra_potent()
    check_importance(X, y, smiles)


if __name__ == "__main__":
    main()
